# Social-RLVR GRPO Training

## Setup
1. Runtime > Change runtime type > T4 GPU
2. Upload `training_data.json` from `artifacts/grpo_dataset/` on your local machine
3. Run all cells in order

## What this does
- Loads Qwen2.5-0.5B-Instruct in 4-bit quantization (fits on T4)
- Applies LoRA adapters for efficient fine-tuning
- Trains using GRPO on your collected (prompt, response, reward) samples
- Saves the trained LoRA weights
- Downloads them back to your machine

In [ ]:
# Cell 1 — Install dependencies
!pip install -q transformers==4.47.0 trl==0.12.0 peft==0.14.0 \
    accelerate==1.2.0 bitsandbytes==0.45.0 datasets==3.2.0 \
    torch==2.5.1 torchvision einops

In [ ]:
# Cell 2 — Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 3 — Upload your training_data.json
from google.colab import files
print('Upload training_data.json from artifacts/grpo_dataset/ on your local machine')
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 4 — Load and inspect the dataset
import json
import pandas as pd

with open('training_data.json') as f:
    raw_data = json.load(f)

print(f'Total samples: {len(raw_data)}')
print(f'Reward distribution:')
rewards = [s['reward'] for s in raw_data]
print(f'  reward=1.0: {rewards.count(1.0)} samples (successful steps)')
print(f'  reward=0.0: {rewards.count(0.0)} samples (unsuccessful steps)')
print(f'\nSample entry:')
print(json.dumps(raw_data[0], indent=2)[:500])

In [ ]:
# Cell 5 — Prepare dataset for GRPO
# GRPO needs: prompt, response (completion), reward
# We group by task episode and assign episode-level reward
# Steps in a successful episode get reward=1, failed episodes get reward=0

from datasets import Dataset

def prepare_grpo_samples(raw_data):
    """
    Convert raw step-level samples to GRPO training format.
    
    GRPO needs groups of completions for the same prompt.
    We treat each (task, repeat) episode as a group.
    Within each group, actions from successful episodes get reward=1.
    """
    samples = []
    
    for item in raw_data:
        # Format as chat messages - this is what the model was trained on
        prompt = [
            {
                'role': 'system',
                'content': 'You are a web browser agent. Given a task and page elements, output exactly one action as JSON.'
            },
            {
                'role': 'user', 
                'content': item['prompt']
            }
        ]
        
        # The completion is what the model actually said
        completion = item['response'] if item['response'] else f'{{"action": "{item["action"]}"}}'
        
        # Reward: use step-level reward
        # Steps that directly led to success get 1.0
        # All other steps get 0.0
        reward = item['reward']
        
        samples.append({
            'prompt': prompt,
            'completion': completion,
            'reward': reward,
            'task_id': item['task_id'],
        })
    
    return samples

grpo_samples = prepare_grpo_samples(raw_data)
dataset = Dataset.from_list(grpo_samples)

# Split 90/10 train/eval
split = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split['train']
eval_dataset = split['test']

print(f'Train samples: {len(train_dataset)}')
print(f'Eval samples: {len(eval_dataset)}')
print(f'Positive reward samples: {sum(1 for s in grpo_samples if s["reward"] > 0)}')

In [ ]:
# Cell 6 — Load model in 4-bit quantization
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

print(f'Loading {MODEL_NAME}...')

# 4-bit quantization config - makes the model fit in T4 VRAM
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',           # NormalFloat4 - best quality
    bnb_4bit_compute_dtype=torch.float16, # compute in fp16
    bnb_4bit_use_double_quant=True,       # double quantization saves more memory
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'left'  # important for causal LM

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Model loaded. Parameters: {model.num_parameters():,}')
print(f'VRAM used: {torch.cuda.memory_allocated() / 1e9:.2f} GB')

In [ ]:
# Cell 7 — Apply LoRA adapters
# LoRA adds small trainable matrices to the model
# Instead of updating all 500M parameters, we only update ~2M adapter parameters
# This makes training feasible on a T4

from peft import prepare_model_for_kbit_training

# Prepare model for k-bit training (enables gradient checkpointing etc)
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,                    # rank - higher = more parameters = better but slower
    lora_alpha=32,           # scaling factor
    lora_dropout=0.05,
    bias='none',
    # target the attention layers - where most of the "reasoning" happens
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Expected output: ~1-2% of parameters are trainable

In [ ]:
# Cell 8 — Define reward function for GRPO
# GRPO needs a reward function it can call during training
# Ours checks if the model output is a valid action format
# AND gives bonus reward if it matches a known successful action

import re

ACTION_RE = re.compile(
    r'^(click|fill|select_option|noop|scroll|press|keyboard_press)\(.*\)$'
)

# Build a lookup of prompt -> best known action from our dataset
# This lets us give reward when the model outputs a known-good action
successful_actions = {}
for item in raw_data:
    if item['reward'] > 0 and item['action'] != 'noop(100)':
        # use first 200 chars of prompt as key (enough to be unique)
        key = item['prompt'][:200]
        successful_actions[key] = item['action']

print(f'Known successful actions: {len(successful_actions)}')


def parse_action_from_response(response: str) -> str | None:
    """Extract action from model response."""
    # try JSON parse
    try:
        data = json.loads(response)
        if isinstance(data, list):
            data = data[0] if data else {}
        action = str(data.get('action', '')).strip()
        if ACTION_RE.match(action):
            return action
    except Exception:
        pass
    # try direct regex
    m = re.search(r'(click|fill|select_option|noop)\([^)]{1,100}\)', response)
    if m and ACTION_RE.match(m.group(0)):
        return m.group(0)
    return None


def reward_function(completions: list[str], prompts: list[list[dict]], **kwargs) -> list[float]:
    """
    Reward function called by GRPOTrainer during training.
    
    Gives reward based on:
    1. +0.3 if the output is valid JSON with an action field
    2. +0.3 if the action is a valid action format (click/fill/etc)
    3. +0.4 if the action matches a known successful action for this prompt
    
    Total possible: 1.0
    """
    rewards = []
    
    for completion, prompt_msgs in zip(completions, prompts):
        reward = 0.0
        
        # get the user message (last user turn)
        user_content = ''
        for msg in reversed(prompt_msgs):
            if msg['role'] == 'user':
                user_content = msg['content']
                break
        
        # reward 1: valid JSON
        try:
            parsed = json.loads(completion)
            if isinstance(parsed, dict) and 'action' in parsed:
                reward += 0.3
        except Exception:
            # partial credit if it looks like JSON
            if '{' in completion and 'action' in completion:
                reward += 0.1
        
        # reward 2: valid action format
        action = parse_action_from_response(completion)
        if action is not None and action != 'noop(100)':
            reward += 0.3
        
        # reward 3: matches known successful action
        prompt_key = user_content[:200]
        if prompt_key in successful_actions:
            if action == successful_actions[prompt_key]:
                reward += 0.4
        
        rewards.append(reward)
    
    return rewards


print('Reward function defined.')
# Test it
test_rewards = reward_function(
    ['{"action": "fill(\\"16\\", \\"TRV-8429-IN\\")"}', 'I will click the button', 'noop(100)'],
    [[{'role': 'user', 'content': 'test'}]] * 3
)
print(f'Test rewards: {test_rewards}')  # should be [0.6, 0.0, 0.3]

In [ ]:
# Cell 9 — Configure and run GRPO training
from trl import GRPOConfig, GRPOTrainer

training_args = GRPOConfig(
    # output
    output_dir='./grpo_rlvr_output',
    
    # training duration
    num_train_epochs=3,
    per_device_train_batch_size=2,     # small batch - T4 memory constraint
    gradient_accumulation_steps=8,     # effective batch size = 2*8 = 16
    
    # GRPO specific
    num_generations=4,                 # how many completions to sample per prompt
                                       # GRPO compares these to compute relative rewards
    max_new_tokens=128,                # max tokens to generate per completion
    temperature=0.8,                   # sampling temperature during training
    
    # learning rate
    learning_rate=1e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    
    # memory optimization
    gradient_checkpointing=True,       # trade compute for memory
    fp16=True,                         # use float16
    optim='paged_adamw_8bit',          # 8-bit optimizer saves VRAM
    
    # logging
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    evaluation_strategy='steps',
    
    # sequence length
    max_prompt_length=1024,            # truncate long prompts
    
    # reproducibility
    seed=42,
    report_to='none',                  # disable wandb
)

trainer = GRPOTrainer(
    model=model,
    args=training_args,
    reward_funcs=reward_function,      # our custom reward function
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

print('GRPOTrainer configured.')
print(f'Training samples: {len(train_dataset)}')
print(f'Starting training...')

trainer.train()

In [ ]:
# Cell 10 — Save the trained LoRA weights
import os

SAVE_PATH = './grpo_rlvr_lora'
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f'Model saved to {SAVE_PATH}')
print('Files saved:')
for f in os.listdir(SAVE_PATH):
    size = os.path.getsize(os.path.join(SAVE_PATH, f))
    print(f'  {f}: {size/1e6:.1f} MB')

In [ ]:
# Cell 11 — Evaluate before vs after on sample prompts
from peft import PeftModel
import torch

def generate_action(model, tokenizer, prompt_text: str, max_new_tokens: int = 128) -> str:
    messages = [
        {'role': 'system', 'content': 'You are a web browser agent. Output exactly one action as JSON.'},
        {'role': 'user', 'content': prompt_text}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return response.strip()


# Test on a few prompts from our dataset
print('=== EVALUATION ON TRAINING SAMPLES ===')
test_samples = [s for s in grpo_samples if s['reward'] > 0][:3]

for i, sample in enumerate(test_samples):
    user_prompt = sample['prompt'][1]['content']  # the user message
    expected_action = sample['completion']
    
    generated = generate_action(model, tokenizer, user_prompt)
    parsed = parse_action_from_response(generated)
    
    print(f'\nSample {i+1}:')
    print(f'  Expected: {expected_action[:80]}')
    print(f'  Generated: {generated[:80]}')
    print(f'  Parsed action: {parsed}')
    print(f'  Match: {generated[:80] == expected_action[:80]}')

In [ ]:
# Cell 12 — Download the trained weights
import shutil
from google.colab import files

# zip the lora folder
shutil.make_archive('grpo_rlvr_lora', 'zip', '.', 'grpo_rlvr_lora')
print('Downloading grpo_rlvr_lora.zip...')
files.download('grpo_rlvr_lora.zip')
print('Done. Extract the zip and place the folder in your project root.')

In [ ]:
# Cell 13 — (OPTIONAL) Push to HuggingFace Hub so you can load it from anywhere
# from huggingface_hub import login
# login(token='your_hf_token_here')
# model.push_to_hub('your-username/social-rlvr-qwen-0.5b-lora')
# tokenizer.push_to_hub('your-username/social-rlvr-qwen-0.5b-lora')
print('Uncomment and fill in your HuggingFace token to push to Hub.')